In [1]:
# Put import statements here
import sys
import subprocess
from pathlib import Path

# Local files/code
import src.data_preprocessing.image_preprocessing as img_pre
import src.data_preprocessing.text_preprocessing as text_pre
import src.data_preprocessing.text_data_exploration as text_explore
from src.util.logger import Logger
from src.data_preprocessing.TextTokenizer import TextTokenizer

# Configuration
from src.config import (
    PEVQA_DATA_PATH, PEVQA_TRAIN_FILE, PEVQA_TEST_FILE,
    PEVQA_VAL_FILE, PEVQA_NA_PEVQA_NA_COLUMN_FILL_PAIRS,
    PEVQA_PEVQA_TEXT_COLUMNS, PEVQA_PEVQA_COLUMNS_TO_REMOVE,
)


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


ImportError: cannot import name 'PEVQA_NA_PEVQA_NA_COLUMN_FILL_PAIRS' from 'src.config' (/home/chris/capstone/PlantQA/src/config.py)

In [ ]:
# Run pytest setup tests to verify the environment/hardware is good to go
#subprocess.run(
#    [sys.executable, "-m", "pytest", "-q", "test/test_setup.py"],
#    check=False,
#)

In [ ]:
# Determine the project root's location, that way file paths will be more consistent
PROJECT_ROOT= Path.cwd()
DATA_DIR= PROJECT_ROOT / "data"

Logger.info(f"Project root absolute path: {PROJECT_ROOT}")

In [ ]:
# load the plantExpertVQA training, validation, and testing datasets
plant_expert_vqa_data_path= DATA_DIR / PEVQA_DATA_PATH
train_data_path= plant_expert_vqa_data_path / PEVQA_TRAIN_FILE
test_data_path= plant_expert_vqa_data_path / PEVQA_TEST_FILE
validation_data_path= plant_expert_vqa_data_path / PEVQA_VAL_FILE


plant_expert_vqa_TRAIN= text_pre.load_csv(train_data_path)
plant_expert_vqa_TEST= text_pre.load_csv(test_data_path)
plant_expert_vqa_VAL= text_pre.load_csv(validation_data_path)

In [ ]:
# Clean and normalize the data (does not tokenize here)

# preprocess (not tokenize) the training, testing, and validation datasets
# NOTE: the stop word removal may be too intense.  We most likely want to fine tune the stopword set, or define our own set
#       Right now it removes words like "what" and "why", which will most likely be bad for a VQA system that answers questions.

# training
text_pre.preprocess_dataframe(
    plant_expert_vqa_TRAIN,
    PEVQA_PEVQA_TEXT_COLUMNS,
    PEVQA_COLUMNS_TO_REMOVE,
    PEVQA_NA_COLUMN_FILL_PAIRS
)

# testing
text_pre.preprocess_dataframe(
    plant_expert_vqa_TEST,
    PEVQA_PEVQA_TEXT_COLUMNS,
    PEVQA_COLUMNS_TO_REMOVE,
    PEVQA_NA_COLUMN_FILL_PAIRS
)

#validation
text_pre.preprocess_dataframe(
    plant_expert_vqa_VAL,
    PEVQA_PEVQA_TEXT_COLUMNS,
    PEVQA_COLUMNS_TO_REMOVE,
    PEVQA_NA_COLUMN_FILL_PAIRS
)


In [ ]:

column_distribution_args= [
    {"column": "crop", "show_counts": False, "figure_size": (10, 5)},
    {"column": "severity", "show_counts": True, "figure_size": (5, 5)},
    {"column": "category", "show_counts": True, "figure_size": (5, 5)},
    {"column": "answer_type", "show_counts": True, "figure_size": (5, 5)},
    {"column": "question_category", "show_counts": False, "figure_size": (10, 5)},
]

# explore the cleaned training dataset
text_explore.explore_data(
    plant_expert_vqa_TRAIN, 
    column_distribution_args, 
    PEVQA_TEXT_COLUMNS, 
    top_n_words=20, 
    name="Plant Expert VQA Training Dataset"
)

In [ ]:
tokenizer= TextTokenizer("distilbert-base-uncased", 128)

encoding= tokenizer.encode_text("hello", None)

Logger.info(encoding)

In [ ]:
head= plant_expert_vqa_TRAIN.iloc()[0]
image_path= head["image_path"]
full_image_path= DATA_DIR / "PlantExpertVQA" / image_path
img= img_pre.load_image(full_image_path)

img_pre.show_image(img)